# The Price is Right

## 第 8 周日程安排

第 1 天：Modal.com 与 SpecialistAgent  
第 2 天：RAG、FrontierAgent、Ensemble Agent  
第 3 天：ScannerAgent、MessengerAgent  
第 4 天：AutonomousPlannerAgent 与 DealAgentFramework  
第 5 天：The Price Is Right 终章

## 基于 80 万条爬取的 Amazon 商品数据集的 RAG（Retrieval Augmented Generation，检索增强生成）

#### 对于我们的第二个 agent，我们会让 OpenAI 估算某个优惠的价格——并给它一些帮助。

我们发现，开箱即用时，LLM 在这方面其实已经很强。

我们也发现，通过微调开源 LLM，可以打败 frontier LLM。

现在我们要尝试 **推理时（inference time）** 技术，而不是训练——通过使用 RAG！

In [ ]:
# 导入

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [ ]:
# 环境

load_dotenv(override=True)
DB = "products_vectorstore"

In [ ]:
# 登录 HuggingFace
# 如果你还没有 HuggingFace 账号，可以在 www.huggingface.co 免费注册一个
# 然后按项目 README 的说明，把 HF_TOKEN 加到你的 .env 文件中

hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

In [ ]:
# LITE_MODE：小数据集调试更快；正式实验用 False

LITE_MODE = False

In [ ]:
# 从 Hub 加载商品数据；后面用 train 填向量库，用 test 做评估

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# 现在创建 Chroma 数据存储

我们将使用免费、开源的向量数据库 Chroma。  
我们会用训练集中的 40 万个商品创建一个 Chroma 数据存储。

In [ ]:
# Chroma：本地持久化向量数据库，路径由 DB 指定

client = chromadb.PersistentClient(path=DB)

# 介绍 SentenceTransformer 编码 LLM

all-MiniLM 是 HuggingFace 上非常有用的模型，它把句子和段落映射到 384 维向量，非常适合语义搜索这类任务。

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

它可以在本地相当快地运行。

作为替代，OpenAI 提供闭源的 Embeddings 模型。与 OpenAI embeddings 相比的好处：
1. 免费且快速！
3. 我们可以在本地运行，数据不会离开我们的机器——如果你在构建个人 RAG，这可能很有用

In [ ]:
# SentenceTransformer：把句子编成稠密向量，便于相似度检索（RAG 的核心）

encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# 传入文本列表，返回向量的 numpy 数组

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

## 有了这些背景，我们来填充 Chroma 数据库

### 通过为 80 万条爬取的商品计算向量

在我的机器上用 GPU 大约需要 30 分钟——对你来说可能更久——可以放心使用 Lite 数据集！

In [ ]:
# 检查 collection 是否存在；若不存在则创建

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

# 让我们可视化向量化后的数据

In [ ]:
# 把这个调到 800_000 并可视化完整数据集非常有趣，
# 但几乎每次都会把我的机器搞崩，所以风险自负！！10_000 是安全的！

MAXIMUM_DATAPOINTS = 10_000

In [ ]:
# 类别名与颜色一一对应，后面画向量散点图用

CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [ ]:
# 准备工作
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
# 我们来试一张 2D 图
# TSNE 是 t-distributed Stochastic Neighbor Embedding 的缩写——一种常用的数据降维技术

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# 创建 2D 散点图
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# 我们来试试 3D！

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# 创建 3D 散点图
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# 看一条测试商品

test[0]

In [ ]:
# 把商品 summary 编码成向量

def vector(item):
    return encoder.encode(item.summary)

In [ ]:
# 在 Chroma 里检索最相似的 5 条商品及其价格（RAG 检索步骤）

def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [ ]:
# 对第一条测试商品试检索

find_similars(test[0])

In [ ]:
# 我们需要通过选择 5 个描述相似的商品，给 GPT-5.1 一些上下文

def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [ ]:
# 打印拼好的上下文，确认相似商品与价格可读

documents, prices = find_similars(test[0])
print(make_context(documents, prices))

In [ ]:
# 组装发给 GPT 的 messages：待估商品 + 相似商品上下文

def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
# 预览完整 user 消息（含 RAG 上下文）

documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

In [ ]:
# 用于 gpt-5-mini 的函数

def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [ ]:
# 我们最喜欢的失真效果器要多少钱？

test[0].price

In [ ]:
# 开始吧！！

gpt_5__1_rag(test[0])

In [ ]:
# 在测试集上评估 RAG 定价的平均误差

evaluate(gpt_5__1_rag, test)

In [ ]:
# 连接第 1 天部署的 Modal 微调定价服务（Specialist）

import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
# Specialist：把 summary 发给微调 Llama 估价格

def specialist(item):
    return pricer.price.remote(item.summary)


In [ ]:
# 从模型回复字符串里解析出数字价格

def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## 把第 6 周的神经网络权重下载到这个目录

文件 `deep_neural_network.pth` 在这里：

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

In [ ]:
# 加载深度神经网络推理器（另一路价格估计）

from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

def deep_neural_network(item):
    return runner.inference(item.summary)

In [ ]:
# 集成（Ensemble）：加权融合 RAG、Specialist、DNN 三路预测
# 权重 0.8/0.1/0.1 可按验证集效果再调

def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1


In [ ]:
# 评估集成模型在测试集上的表现

evaluate(ensemble, test)

In [ ]:
# 打开日志，观察后续 Agent 调用细节

root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# FrontierAgent：封装「向量检索 + 前沿模型 RAG 定价」

from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

In [ ]:
# 再试另一款麦克风描述

agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
# NeuralNetworkAgent：用训练好的神经网络估价格

from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()


In [ ]:
# 用同一描述对比不同 Agent 的报价

agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
# EnsembleAgent：把多路估价打包成一个 Agent 接口

from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

In [ ]:
# 最终集成 Agent 询价

agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")